In [1]:
import torch
import torchvision.models as models
import torch.nn as nn

import numpy as np


In [2]:
class IMUOnlyClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(IMUOnlyClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # Get the last time step
        return out

In [3]:
checkpoint = torch.load('/Users/sanjivjha/Developer/ubicomp-capstone/model/imu_only_rnn_imwsha_100.pth')


In [4]:
input_size = 6  # For example, 2*(3 axes for gyro and 3 for accel)
hidden_size = 35  # Example size, can be tuned
num_layers = 2  # Number of LSTM layer
CLASSES = ["drinking", "packaging", "resting", "lifting", "cutting", "using-computer-mouse"]
num_classes = len(CLASSES)
torch.manual_seed(42)
model = IMUOnlyClassifier(input_size, hidden_size, num_layers, num_classes)

In [5]:
model.eval()

IMUOnlyClassifier(
  (lstm): LSTM(6, 35, num_layers=2, batch_first=True)
  (fc): Linear(in_features=35, out_features=6, bias=True)
)

In [6]:
import pandas as pd
input_df = pd.read_csv("/Users/sanjivjha/Developer/ubicomp-capstone/pacakging-scenario-data/out.csv")
input_df = input_df.loc[:, ~input_df.columns.str.contains('^Unnamed')]
input_df.head()

,ax1,ay1,az1,gx1,gy1,gz1,temperature,timestamp,ax2,ay2,az2,gx2,gy2,gz2
0,3873.259080,-7771.304469,-110.936223,-7298.429725,-4186.429420,3061.570252,29.000,1.747745e+09,1597.283767,-4154.439848,-58.442177,-6375.378310,-5065.628894,238.459112
1,1458.218902,-4102.852987,-258.115603,-6258.237004,-4977.078758,148.864649,29.125,1.747745e+09,1597.283767,-4154.439848,-58.442177,-6375.378310,-5065.628894,238.459112
2,3.628252,1494.272413,48.251777,-5817.556721,-4504.386682,283.899428,29.125,1.747745e+09,1253.527259,-2480.549899,-201.001613,-6122.498678,-4556.003768,763.558774
3,-91.962133,1518.255234,-359.360502,-6764.190889,-4490.952219,316.175018,29.125,1.747745e+09,-433.659858,2578.873679,-89.909438,-6932.625321,-4544.930274,490.449966
4,-901.329805,3803.195780,236.961450,-6805.622784,-4632.670799,522.084996,29.125,1.747745e+09,-433.659858,2578.873679,-89.909438,-6932.625321,-4544.930274,490.449966



# for xgboost 
import torch.nn.functional as F
output_data = pd.DataFrame([], columns=['Activity'])
for i in input_df.index.values.tolist():
    imu_data_tensor = torch.tensor(input_df[['ax1','ay1','az1','gx1','gy1','gz1','ax2','ay2','az2','gx2','gy2','gz2']].iloc[i].values, dtype=torch.float32)
    imu_data_tensor = imu_data_tensor.view(1, 1, -1)
    output =model (imu_data_tensor)
    probabilities = F.softmax(output, dim=1)
    _, predicted = torch.max(probabilities, dim=1)
    confidence_scores = torch.gather(probabilities, 1, predicted.unsqueeze(1))
    output_data.loc[len(output_data.index)] = [CLASSES[predicted.numpy()[0]]]
    print( 'Activity = {}, confidence = {}'.format(CLASSES[predicted.item()], confidence_scores.item()*100))

#output_data.to_csv('../collected/final_output.csv')

In [7]:
# for LSTM create sequence of observations and then predict activities
timestamps = input_df['timestamp'].values
temperature = input_df['temperature'].values
features = input_df.drop(columns=['ax2','ay2','az2', 'gx2','gy2','gz2','timestamp', 'temperature']).values

def create_sequences(data, timestamps, temperature, seq_length):
    sequences = []
    seq_timestamps = []
    seq_temperature =[]
    for i in range(len(data) - seq_length + 1):
        seq = data[i:i + seq_length]
        sequences.append(seq)
        # use last timestamp in window
        seq_timestamps.append(timestamps[i + seq_length - 1])
        seq_temperature.append(temperature[i + seq_length - 1])
    return np.array(sequences), np.array(seq_timestamps), np.array(seq_temperature)

seq_length = 200
X_test, seq_timestamps, seq_temperature = create_sequences(features, timestamps, temperature, seq_length)

In [8]:
# predict
# Convert test data to PyTorch tensor
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

# Make predictions
with torch.no_grad():
    outputs = model(X_test_tensor)

# Get predicted classes and confidence
probabilities = torch.softmax(outputs, dim=1)
confidence_scores, predicted_classes = torch.max(probabilities, dim=1)

# Map class indices to labels
activity_labels = ["drinking", "packaging", "resting", "lifting", "cutting", "using-computer-mouse"]
predicted_activities = [activity_labels[i] for i in predicted_classes.numpy()]

In [9]:
# Create DataFrame
output_df = pd.DataFrame({
    'timestamp': pd.to_datetime(seq_timestamps, unit='s'),
    'temperature': pd.to_numeric(seq_timestamps),
    'predicted_activity': predicted_activities,
    'confidence': confidence_scores.numpy()
})

# Display and save
print(output_df)


                         timestamp   temperature predicted_activity  \
0    2025-05-20 12:40:36.460468054  1.747745e+09          packaging   
1    2025-05-20 12:40:36.550448895  1.747745e+09          packaging   
2    2025-05-20 12:40:36.618825912  1.747745e+09          packaging   
3    2025-05-20 12:40:36.685544968  1.747745e+09            cutting   
4    2025-05-20 12:40:36.775559902  1.747745e+09            cutting   
...                            ...           ...                ...   
1353 2025-05-20 12:42:20.498948097  1.747745e+09          packaging   
1354 2025-05-20 12:42:20.545063972  1.747745e+09          packaging   
1355 2025-05-20 12:42:20.633374929  1.747745e+09          packaging   
1356 2025-05-20 12:42:20.701143980  1.747745e+09          packaging   
1357 2025-05-20 12:42:20.791385174  1.747745e+09          packaging   

      confidence  
0       0.195440  
1       0.195055  
2       0.195176  
3       0.196030  
4       0.196246  
...          ...  
1353    0.1962

In [10]:
output_df.to_csv('/Users/sanjivjha/Developer/ubicomp-capstone/pacakging-scenario-data/predicted_activities_with_timestamps.csv', index=False)

import data_plotter as dpl

dpl.create_plots(input_df, '/Users/sanjivjha/Developer/ubicomp-capstone/custom_IMSHA_Dataset/')